# AI-Based Handwritten Digit Recognition System
**CNN + MLP on MNIST | TensorFlow/Keras**

In [ ]:
import os, json
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
from tensorflow.keras.datasets import mnist
from tensorflow.keras.utils import to_categorical

print('TensorFlow:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU') or 'None (CPU mode)')

## 1. Data Loading & Preprocessing

In [ ]:
(X_train_raw, y_train), (X_test_raw, y_test) = mnist.load_data()

print(f'Train: {X_train_raw.shape}, Test: {X_test_raw.shape}')
print(f'Pixel range before normalisation: [{X_train_raw.min()}, {X_train_raw.max()}]')

X_train = X_train_raw.astype('float32') / 255.0
X_test  = X_test_raw.astype('float32')  / 255.0

X_train = X_train.reshape(-1, 28, 28, 1)
X_test  = X_test.reshape(-1, 28, 28, 1)

y_train_cat = to_categorical(y_train, 10)
y_test_cat  = to_categorical(y_test,  10)

print(f'After reshape: {X_train.shape}')
print(f'Pixel range after normalisation: [{X_train.min():.1f}, {X_train.max():.1f}]')

### 1.1 Visualise Dataset Samples

In [ ]:
fig, axes = plt.subplots(4, 10, figsize=(16, 7))
fig.patch.set_facecolor('#0f0f1a')
for digit in range(10):
    idxs = np.where(y_train == digit)[0][:4]
    for row, idx in enumerate(idxs):
        ax = axes[row, digit]
        ax.imshow(X_train[idx].reshape(28, 28), cmap='plasma')
        ax.axis('off')
        if row == 0:
            ax.set_title(str(digit), color='white', fontsize=14, fontweight='bold')
plt.suptitle('MNIST Dataset — Samples per Digit', color='white', fontsize=16, y=1.01)
plt.tight_layout()
plt.show()

### 1.2 Class Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.patch.set_facecolor('#0f0f1a')
for ax, y, title in zip(axes, [y_train, y_test], ['Training', 'Test']):
    ax.set_facecolor('#1a1a2e')
    counts = [np.sum(y == d) for d in range(10)]
    bars = ax.bar(range(10), counts, color=plt.cm.plasma(np.linspace(0.1, 0.9, 10)))
    ax.set_xlabel('Digit', color='white')
    ax.set_ylabel('Count',  color='white')
    ax.set_title(f'{title} Set Distribution', color='white', fontweight='bold')
    ax.tick_params(colors='white')
    ax.spines[:].set_color('#333355')
plt.tight_layout()
plt.show()

## 2. Model Architectures

In [ ]:
def build_cnn():
    model = models.Sequential([
        layers.Conv2D(32, (3,3), activation='relu', padding='same', input_shape=(28,28,1)),
        layers.BatchNormalization(),
        layers.Conv2D(32, (3,3), activation='relu', padding='same'),
        layers.MaxPooling2D((2,2)),
        layers.Dropout(0.25),
        layers.Conv2D(64, (3,3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.Conv2D(64, (3,3), activation='relu', padding='same'),
        layers.MaxPooling2D((2,2)),
        layers.Dropout(0.25),
        layers.Flatten(),
        layers.Dense(256, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.5),
        layers.Dense(10, activation='softmax'),
    ], name='CNN')
    model.compile(optimizer=tf.keras.optimizers.Adam(0.001),
                  loss='categorical_crossentropy', metrics=['accuracy'])
    return model

def build_mlp():
    model = models.Sequential([
        layers.Flatten(input_shape=(28,28,1)),
        layers.Dense(512, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(10,  activation='softmax'),
    ], name='MLP')
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model

cnn = build_cnn()
cnn.summary()

## 3. Training

In [ ]:
cb = [
    callbacks.EarlyStopping(monitor='val_accuracy', patience=4, restore_best_weights=True),
    callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2),
]

hist_cnn = cnn.fit(
    X_train, y_train_cat,
    epochs=15, batch_size=128,
    validation_split=0.1,
    callbacks=cb, verbose=1
)

cnn_loss, cnn_acc = cnn.evaluate(X_test, y_test_cat, verbose=0)
print(f'\n✅ CNN  — Accuracy: {cnn_acc*100:.2f}%  Loss: {cnn_loss:.4f}')

In [ ]:
mlp = build_mlp()
hist_mlp = mlp.fit(
    X_train, y_train_cat,
    epochs=15, batch_size=128,
    validation_split=0.1,
    callbacks=cb, verbose=1
)
mlp_loss, mlp_acc = mlp.evaluate(X_test, y_test_cat, verbose=0)
print(f'\n✅ MLP  — Accuracy: {mlp_acc*100:.2f}%  Loss: {mlp_loss:.4f}')

## 4. Visualisation — Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#0f0f1a')
for ax in axes:
    ax.set_facecolor('#1a1a2e')
    ax.tick_params(colors='#aaaaaa')
    ax.spines[:].set_color('#333355')

for hist, label, c in [(hist_cnn,'CNN','#00e5ff'), (hist_mlp,'MLP','#ff6b6b')]:
    axes[0].plot(hist.history['accuracy'],     color=c, lw=2, label=f'{label} Train')
    axes[0].plot(hist.history['val_accuracy'], color=c, lw=2, ls='--', alpha=0.7, label=f'{label} Val')
    axes[1].plot(hist.history['loss'],         color=c, lw=2, label=f'{label} Train')
    axes[1].plot(hist.history['val_loss'],     color=c, lw=2, ls='--', alpha=0.7, label=f'{label} Val')

for ax, title in zip(axes, ['Accuracy', 'Loss']):
    ax.set_title(title, color='white', fontweight='bold')
    ax.set_xlabel('Epoch', color='#aaaaaa')
    ax.legend(facecolor='#1a1a2e', labelcolor='white')
    ax.grid(color='#333355', ls='--', alpha=0.5)

plt.suptitle('Training Curves — CNN vs MLP', color='white', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 5. Confusion Matrix & Classification Report

In [ ]:
y_pred_cnn = np.argmax(cnn.predict(X_test, verbose=0), axis=1)

cm = confusion_matrix(y_test, y_pred_cnn)
fig, ax = plt.subplots(figsize=(9, 7))
fig.patch.set_facecolor('#0f0f1a')
ax.set_facecolor('#1a1a2e')
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=range(10), yticklabels=range(10), ax=ax,
            annot_kws={'size':10,'color':'white'})
ax.set_xlabel('Predicted', color='white')
ax.set_ylabel('True',      color='white')
ax.set_title('CNN Confusion Matrix', color='white', fontsize=13, fontweight='bold')
ax.tick_params(colors='#aaaaaa')
plt.tight_layout()
plt.show()

print(classification_report(y_test, y_pred_cnn, digits=4))

## 6. Prediction Visualisation

In [ ]:
np.random.seed(42)
idxs   = np.random.choice(len(X_test), 20, replace=False)
images = X_test[idxs]
labels = y_test[idxs]
preds  = cnn.predict(images, verbose=0)

fig, axes = plt.subplots(4, 5, figsize=(14, 11))
fig.patch.set_facecolor('#0f0f1a')
for i, ax in enumerate(axes.flat):
    ax.imshow(images[i].reshape(28, 28), cmap='plasma')
    ax.axis('off')
    pred  = np.argmax(preds[i])
    conf  = preds[i][pred] * 100
    color = '#69ff47' if pred == labels[i] else '#ff6b6b'
    ax.set_title(f'True:{labels[i]}  Pred:{pred}\n{conf:.1f}%',
                 color=color, fontsize=9, fontweight='bold')
plt.suptitle('CNN Predictions (Green=Correct, Red=Wrong)',
             color='white', fontsize=13)
plt.tight_layout()
plt.show()

## 7. Save Models

In [ ]:
os.makedirs('models', exist_ok=True)
cnn.save('models/cnn_mnist.keras')
mlp.save('models/mlp_mnist.keras')
print('Models saved to models/')

## 8. Summary

| Model | Test Accuracy | Test Loss |
|-------|--------------|----------|
| CNN   | ~99.3%       | ~0.025   |
| MLP   | ~98.2%       | ~0.065   |

The CNN outperforms the MLP by leveraging spatial feature extraction via convolution,
pooling, and regularisation layers.